## Setting up search

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

In [2]:
df_ground_truth.head(5)

,question,document
0,I just found this course late — am I still all...,74eb249bbf
1,Is it too late to start the course if I missed...,74eb249bbf
2,Can I enroll after the course has already star...,74eb249bbf
3,"If I join the course now, can I still get a ce...",74eb249bbf
4,What’s the deadline for getting the certificat...,74eb249bbf


In [4]:
ground_truth[3]

{'question': 'If I join the course now, can I still get a certificate?',
 'document': '74eb249bbf'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
        
documents = documents_llm
index = build_index(documents)

In [10]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

## Collecting relevance data

In [8]:
q = ground_truth[0]
q

{'question': 'I just found this course late — am I still allowed to join in now?',
 'document': '74eb249bbf'}

In [12]:
doc_id = q['document']
results = text_search(query=q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I st

In [13]:
## Compare retrieved doc IDs with correct doc ID
for d in results:
    print(f'{d['id']} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
85384a18e5 == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
c2903069a0 == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [14]:
relevance = []

for d in results:
    relevance.append(int(d['id'] == doc_id))
    
relevance

[1, 0, 0, 0, 0]

In [20]:
## Wrap it in the function
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [16]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

I just found this course late — am I still allowed to join in now?


[1, 0, 0, 0, 0]

In [17]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)

Where do I follow the LLM Zoomcamp syllabus, homework, and deadlines in one place?


[1, 0, 0, 0, 0]

In [19]:
from tqdm.auto import tqdm

## Do the same thing for all ground truth questions
def compute_relevance_total_text(ground_truth):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)
        
    return relevance_total

In [21]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)


  0%|          | 0/15 [00:00<?, ?it/s]

In [22]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [23]:
## Make the relevance function generic (the logic the same, only search function changes)
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [24]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
        
    return relevance_total

In [25]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [26]:
# Run for all ground truth questions
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

## Evaluation Metrics
### Hit Rate

In [27]:
sample = relevance_total[:15]
sample

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0]]

In [28]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt += 1
        
cnt

10

In [29]:
cnt / len(sample) ## Hit Rate

0.6666666666666666

In [30]:
## Put into a function
def hit_rate(relevance):
    cnt = 0
    
    for line in relevance:
        if 1 in line:
            cnt +=1
            
    return cnt/len(relevance)

In [31]:
hit_rate(sample)

0.6666666666666666

### Mean Reciprocal Rank (MRR)

In [33]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1/(rank+1)
            break

total_score

7.283333333333333

In [34]:
total_score/len(sample)

0.4855555555555556

In [35]:
## Put into the function
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [36]:
mrr(sample)

0.4855555555555556

In [37]:
## Wrap metrics in a reusable evaluation function
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [38]:
evaluate(ground_truth, text_search)

  0%|          | 0/590 [00:00<?, ?it/s]

{'hit_rate': 0.8389830508474576, 'mrr': 0.7305084745762708}

## Search Parameter Tuning

In [39]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [40]:
for boost in [0.5, 1, 3, 5, 10]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    
    print(f'boost={boost}: {result}')

  0%|          | 0/590 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8728813559322034, 'mrr': 0.7905367231638416}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=1: {'hit_rate': 0.8949152542372881, 'mrr': 0.7870338983050843}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=3: {'hit_rate': 0.8389830508474576, 'mrr': 0.7305084745762708}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=5: {'hit_rate': 0.8220338983050848, 'mrr': 0.6974011299435025}


  0%|          | 0/590 [00:00<?, ?it/s]

boost=10: {'hit_rate': 0.7898305084745763, 'mrr': 0.6577683615819206}


In [41]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [42]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/590 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/590 [00:00<?, ?it/s]

In [43]:
df_result = pd.DataFrame(results)
df_result.sort_values(by="mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
4,1.0,2.0,0.2,0.947458,0.864322
20,2.0,4.0,0.5,0.947458,0.864040
3,1.0,2.0,0.1,0.952542,0.863305
35,5.0,10.0,0.5,0.952542,0.863305
19,2.0,4.0,0.2,0.952542,0.863305
7,1.0,4.0,0.2,0.957627,0.861356
8,1.0,4.0,0.5,0.955932,0.860989
18,2.0,4.0,0.1,0.952542,0.860819
34,5.0,10.0,0.2,0.952542,0.860254
33,5.0,10.0,0.1,0.950847,0.858333
